# Lecture 9: Open Economy Macroeconomics I

**Macroeconomics B -- Chapter 23**

By the end of this notebook you will be able to:
1. Compute the domestic interest rate implied by uncovered interest parity (UIP).
2. Visualise why a credible peg pins the domestic rate to the foreign rate (trilemma).
3. Measure how inflation differentials shift the real exchange rate using live HICP data.
4. Simulate when a real depreciation improves or worsens the trade balance (J-curve).
5. Trace the full AD adjustment after a negative demand shock and after a positive export boom.


**Table of contents**<a id='toc0_'></a>
- 1. [Notebook map and calibration](#toc1_)
- 2. [Uncovered interest parity](#toc2_)
  - 2.1. [Try it yourself](#toc2_1_)
- 3. [The trilemma: peg versus float](#toc3_)
  - 3.1. [Try it yourself](#toc3_1_)
- 4. [Real exchange rates and Danish inflation differentials](#toc4_)
  - 4.1. [Try it yourself](#toc4_1_)
- 5. [Marshall-Lerner and the J-curve](#toc5_)
  - 5.1. [Try it yourself](#toc5_1_)
- 6. [Open-economy AD adjustment under a peg](#toc6_)
  - 6.1. [Try it yourself](#toc6_1_)
- 7. [Group exercise](#toc7_)
- 8. [Summary](#toc8_)


In [ ]:
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

# Global plot settings -- font sizes set individually so tick labels stay readable
plt.rcParams.update({
    'axes.grid': True, 'grid.color': 'black',
    'grid.alpha': 0.25, 'grid.linestyle': '--',
    'font.size': 12,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
})

rng    = np.random.default_rng(2025)
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']


## 1. <a id='toc1_'></a>[Notebook map and calibration](#toc0_)

*Corresponds to slides: "Roadmap for Today" (frame 3) and "Learning Goals" (frame 4).*

| Section | Model object | Key equation | Slide frames |
|---|---|---|---|
| 2. UIP | UIP schedule | $i_t = i_t^f + \Delta e^e_{t+1}$ | 9-11 |
| 3. Trilemma | peg vs float | credible peg: $\Delta e^e \approx 0$ | 12-13 |
| 4. Real exchange rate | competitiveness index | $\Delta e^r_t = \Delta e_t + \pi^f_t - \pi_t$ | 15, 17 |
| 5. Marshall-Lerner | J-curve | $\varepsilon_X + \varepsilon_M > 1$ | 21-22 |
| 6. AD adjustment | fixed-rate transition | recession $\to$ low $\pi$ $\to$ real depreciation $\to$ recovery | 28, 32 |

Parameters are deliberately simple. Edit them once in the calibration cell and the changes propagate everywhere.


In [ ]:
# === Baseline parameters (edit here for all experiments) ===
par = {
    'T':         40,
    'i_f':       0.03,    # foreign nominal interest rate
    'sigma_dep': 0.012,   # volatility of expected depreciation (float)
    'theta':     0.70,    # regressive-expectations strength (0=random walk, 1=full reversion)
    'rho_de':    0.65,    # persistence of realised nominal depreciation
    'beta1':     0.80,    # net-export sensitivity to the real exchange rate
    'beta2':     0.60,    # demand sensitivity to the real interest rate
    'gamma':     0.50,    # AS slope (wage/price flexibility)
    'z_shock':   -1.00,   # negative demand shock in period 0
}


# ── UIP helpers ────────────────────────────────────────────────────────────────

def uip_rate(i_f, expected_depreciation):
    # log-linear UIP: i = i_f + Delta_e_exp
    return i_f + expected_depreciation


def simulate_expected_depreciation(T, sigma, rng):
    # iid expected-depreciation shocks under a float
    dep    = np.empty(T)
    shocks = rng.normal(0.0, sigma, size=T)
    for t in range(T):
        dep[t] = shocks[t]
    return dep


def simulate_regressive_expectations(T, rho_de, theta, sigma_dep, rng):
    # regressive expectations: weak today implies expected appreciation tomorrow
    # theta controls mean-reversion strength
    de     = np.empty(T)
    de_exp = np.empty(T)
    shocks = rng.normal(0.0, sigma_dep, size=T)
    de[0]     = shocks[0]
    de_exp[0] = -theta * de[0]
    for t in range(1, T):
        de[t]     = rho_de * de[t-1] + shocks[t]
        de_exp[t] = -theta * de[t]
    return de, de_exp


# ── FRED / HICP helpers ────────────────────────────────────────────────────────

def fallback_hicp_data():
    # compact fallback calibrated to actual Eurostat readings (offline use)
    dates = pd.date_range('2020-01-01', '2026-04-01', freq='MS')
    ann = pd.DataFrame({
        'dk': np.interp(np.arange(len(dates)), [0,12,24,36,48,60,75],
                        [0.4, 1.8, 8.5, 3.3, 1.3, 1.6, 1.2]),
        'ea': np.interp(np.arange(len(dates)), [0,12,24,36,48,60,75],
                        [0.3, 2.6, 8.4, 5.4, 2.4, 2.2, 3.0]),
    }, index=dates)
    dk = np.empty(len(ann)); ea = np.empty(len(ann))
    dk[0] = 85.0; ea[0] = 84.0
    for t in range(1, len(ann)):
        dk[t] = dk[t-1] * (1.0 + ann['dk'].iloc[t] / 1200.0)
        ea[t] = ea[t-1] * (1.0 + ann['ea'].iloc[t] / 1200.0)
    return {
        'CP0000DKM086NEST':   pd.Series(dk, index=dates),
        'CP0000EZ19M086NEST': pd.Series(ea, index=dates),
    }


def fetch_fred_series(series_id, fallback):
    # download FRED CSV; fall back to sample data when offline
    url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}'
    try:
        r = requests.get(url, timeout=10); r.raise_for_status()
        df = pd.read_csv(io.StringIO(r.text))
        df['observation_date'] = pd.to_datetime(df['observation_date'])
        s   = pd.to_numeric(df[series_id], errors='coerce')
        out = pd.Series(s.values, index=df['observation_date']).dropna()
        return out, f'FRED live: {series_id}'
    except Exception as err:
        return fallback[series_id].copy(), f'fallback used: {series_id} ({type(err).__name__})'


def yoy_inflation(idx):
    # year-on-year inflation from a monthly price index
    return 100.0 * (idx / idx.shift(12) - 1.0)


# ── Marshall-Lerner / J-curve ──────────────────────────────────────────────────

def simulate_j_curve(T, depreciation,
                     eps_x_start, eps_m_start,
                     eps_x_long, eps_m_long, speed):
    # price effect immediate; elasticities converge exponentially to long-run values
    # trade-balance change = depreciation * (eps_X + eps_M - 1)
    elasticity_sum = np.empty(T)
    trade_balance  = np.empty(T)
    for t in range(T):
        w = 1.0 - np.exp(-speed * t)
        eps_x = eps_x_start + w * (eps_x_long - eps_x_start)
        eps_m = eps_m_start + w * (eps_m_long - eps_m_start)
        elasticity_sum[t] = eps_x + eps_m
        trade_balance[t]  = depreciation * (elasticity_sum[t] - 1.0)
    return elasticity_sum, trade_balance


# ── Open-economy AD adjustment ─────────────────────────────────────────────────

def simulate_fixed_rate_adjustment(T, beta1, gamma, z0):
    # fixed nominal exchange rate, deviations from long-run equilibrium:
    #   yhat_t  = (beta1 * er_{t-1} + z_t) / (1 + beta1 * gamma)
    #   pihat_t = gamma * yhat_t
    #   er_t    = er_{t-1} - pihat_t
    yhat  = np.empty(T); pihat = np.empty(T)
    er    = np.empty(T); z     = np.empty(T)
    z[:]  = 0.0;          z[0]  = z0
    er_lag = 0.0
    for t in range(T):
        yhat[t]  = (beta1 * er_lag + z[t]) / (1.0 + beta1 * gamma)
        pihat[t] = gamma * yhat[t]
        er[t]    = er_lag - pihat[t]
        er_lag   = er[t]
    return pd.DataFrame({'period': np.arange(T),
                         'yhat': yhat, 'pihat': pihat, 'er': er})


## 2. <a id='toc2_'></a>[Uncovered interest parity](#toc0_)

*Corresponds to slides: "UIP: The Exact No-Arbitrage Condition" (frame 9), "UIP: The Log-Linear Version" (frame 10), and "UIP Mechanism" (frame 11).*

Log-linear UIP:

$$i_t = i_t^f + \Delta e^e_{t+1}$$

The **domestic nominal interest rate** must equal the **foreign rate** plus **expected depreciation**. If investors expect the domestic currency to weaken, they demand a higher rate to hold domestic bonds. The table below shows the implied domestic rate for five levels of expected depreciation, holding $i^f = 3\%$.


In [ ]:
# === UIP table ===
exp_dep = np.array([-0.02, 0.00, 0.01, 0.02, 0.05])
i_dom   = uip_rate(par['i_f'], exp_dep)

uip_table = pd.DataFrame({
    'Expected depreciation (%)':  (100 * exp_dep).round(1),
    'Foreign interest rate (%)':  100 * par['i_f'],
    'UIP domestic rate (%)':      (100 * i_dom).round(2),
})
uip_table


In [ ]:
# === UIP schedule ===
dep_grid = np.linspace(-0.04, 0.07, 200)
i_grid   = uip_rate(par['i_f'], dep_grid)

fig = plt.figure(figsize=(7.5, 4.2))
ax  = fig.add_subplot(1, 1, 1)

ax.plot(100 * dep_grid, 100 * i_grid, lw=2, ls='-', label='UIP schedule')
ax.axhline(100 * par['i_f'], lw=1, ls='--', color='grey',
           label=f"Foreign rate $i^f = {100*par['i_f']:.0f}\\%$")
ax.axvline(0.0, lw=1, ls=':', color='grey')
ax.scatter(100 * exp_dep, 100 * i_dom, zorder=5, s=55, color=colors[0])

ax.set_title('UIP: domestic rate rises one-for-one with expected depreciation')
ax.set_xlabel(r'Expected depreciation $\Delta e^e$ (percent)')
ax.set_ylabel(r'Domestic nominal rate $i$ (percent)')
ax.legend(loc='upper left')
fig.tight_layout()


**What is done:** The equation $i = i^f + \Delta e^e$ is plotted as a schedule in the $(\Delta e^e,\, i)$ plane. Five specific values from the table are marked as dots.

**Why it is useful:** The schedule shows why a **credible peg** pins the domestic rate. With $\Delta e^e \approx 0$, the equilibrium is $i \approx i^f$. Danmarks Nationalbank's eight rate cuts in 2024 each matched the ECB cut step-by-step because holding $i = i^f$ is the operational content of "credible peg."

**How to interpret the result:** The slope is exactly 1: every 1 percentage-point rise in expected depreciation requires a 1 percentage-point higher domestic rate. The dashed line shows $i^f$; its intersection with the schedule at $\Delta e^e = 0$ is the **peg equilibrium**. Moving right along the schedule corresponds to a country where the peg is losing credibility.


### 2.1. <a id='toc2_1_'></a>[Try it yourself](#toc0_)

**Task A.** Change `par['i_f']` to `0.04` (the ECB deposit rate in mid-2023) and rerun the table and plot. Does the **slope** of the UIP schedule change, or only its **level**? What Danish overnight rate does UIP predict while the ECB held at 4.0%?

**Task B.** Use `uip_rate` to compute the domestic rate Danmarks Nationalbank would need to set if markets priced in a 2% expected depreciation. Compare this to the actual 2015 episode, where the pressure was toward *appreciation* (expected depreciation was negative) and the Nationalbank cut its deposit rate to $-0.75\%$ to prevent the krone from strengthening.

**Task C.** Set `par['i_f'] = -0.005` (negative ECB deposit rate, 2016-2022). At what expected depreciation does the UIP-implied domestic rate cross zero? What does this tell you about the constraints on using negative rates to defend a peg when markets expect appreciation?


In [ ]:
# write your code here


## 3. <a id='toc3_'></a>[The trilemma: peg versus float](#toc0_)

*Corresponds to slide: "The Macroeconomic Trilemma" (frame 13).*

A credible fixed exchange rate forces $\Delta e^e \approx 0$, which through UIP locks $i \approx i^f$. Under a float, expected depreciation varies and opens a wedge between domestic and foreign rates.

The first plot below draws random expected-depreciation shocks and shows the implied domestic rate under both regimes. The second shows **regressive expectations**: when the currency is weak today, markets expect future appreciation.


In [ ]:
# === Credible peg versus float: interest-rate volatility ===
T         = par['T']
dep_float = simulate_expected_depreciation(T, par['sigma_dep'], rng)
dep_peg   = np.zeros(T)

i_float = uip_rate(par['i_f'], dep_float)
i_peg   = uip_rate(par['i_f'], dep_peg)
period  = np.arange(T)

fig = plt.figure(figsize=(8.5, 4.2))
ax  = fig.add_subplot(1, 1, 1)

ax.plot(period, 100 * i_float, lw=2, ls='-',  color=colors[0],
        label='Float: expected depreciation moves')
ax.plot(period, 100 * i_peg,   lw=2, ls='--', color=colors[1],
        label=r'Credible peg: $\Delta e^e \approx 0$')
ax.axhline(100 * par['i_f'], lw=1, ls=':', color='grey', label=r'Foreign rate $i^f$')

ax.set_title('UIP domestic rate: peg locks it, float opens a wedge')
ax.set_xlabel('Period')
ax.set_ylabel('Nominal interest rate (percent)')
ax.legend(loc='upper right')
fig.tight_layout()


**What is done:** The UIP equation is applied to two paths for $\Delta e^e$: a flat zero path (peg) and a stochastic iid path (float).

**Why it is useful:** This is the **trilemma in one figure**. Under the peg, the domestic rate equals $i^f$ every period. Under the float, the central bank gains independence but pays in exchange-rate volatility. For Denmark, choosing the peg means accepting that domestic mortgage rates and corporate borrowing costs are essentially determined in Frankfurt, not Copenhagen.

**How to interpret the result:** The dashed line (peg) sits flat at $i^f$ throughout. The solid line (float) fluctuates around the same level, with volatility set by `sigma_dep`. Increasing `sigma_dep` makes the float line more volatile, widening the gap between the two regimes.


In [ ]:
# === Regressive expectations under a float ===
de, de_exp = simulate_regressive_expectations(
    T, par['rho_de'], par['theta'], par['sigma_dep'], rng)
i_reg = uip_rate(par['i_f'], de_exp)

fig = plt.figure(figsize=(8.5, 4.2))
ax  = fig.add_subplot(1, 1, 1)

ax.plot(period, 100 * de,     lw=2, ls='-',  color=colors[0],
        label=r'Realised depreciation $\Delta e_t$')
ax.plot(period, 100 * de_exp, lw=2, ls='--', color=colors[1],
        label=r'Expected depreciation $\Delta e^e_{t+1}$')
ax.plot(period, 100 * i_reg,  lw=1.5, ls=':', color=colors[2],
        label=r'UIP domestic rate $i_t$')
ax.axhline(100 * par['i_f'], lw=1, ls=':', color='grey', label=r'Foreign rate $i^f$')

ax.set_title('Regressive expectations: weak today implies expected appreciation tomorrow')
ax.set_xlabel('Period')
ax.set_ylabel('Percent')
ax.legend(loc='upper right')
fig.tight_layout()


**What is done:** Under regressive expectations ($\Delta e^e_t = -\theta \cdot \Delta e_t$), a currently weak currency is expected to partly reverse. The parameter `theta` controls the strength of mean reversion.

**Why it is useful:** **Regressive expectations** are empirically plausible and help close floating-rate models. They also explain the 2015 Danish episode: once the Nationalbank made peg-defence credible through rate cuts and FX purchases, expected depreciation collapsed, the domestic rate premium disappeared, and FX market pressure dissipated.

**How to interpret the result:** When realised depreciation is positive (currency weak, solid above zero), expected future depreciation is negative (dashed below zero, markets anticipate recovery). The dotted UIP rate then dips below $i^f$: a weak currency that is expected to strengthen means holding domestic assets is even more attractive, consistent with UIP requiring a **lower** rate to restore indifference.


### 3.1. <a id='toc3_1_'></a>[Try it yourself](#toc0_)

**Task A.** Reduce `par['sigma_dep']` from `0.012` to `0.003`. How does domestic interest-rate volatility change under the float? At roughly what sigma does the float become visually indistinguishable from the peg?

**Task B.** Increase `par['theta']` from `0.70` to `0.97` (near-perfect regressive expectations). How much does the expected depreciation path smooth out relative to the realised path? At exactly `theta = 1.0`, what is $\Delta e^e_t$ as a function of $\Delta e_t$?

**Task C.** Set `par['rho_de'] = 0.95` (highly persistent depreciation). This mimics sustained speculative pressure on a currency. How does the domestic interest-rate path respond? What does high persistence imply for FX reserve depletion when the central bank must intervene?


In [ ]:
# write your code here


## 4. <a id='toc4_'></a>[Real exchange rates and Danish inflation differentials](#toc0_)

*Corresponds to slides: "Real Exchange Rate Dynamics" (frame 15) and "Denmark and the Euro Area: A Recent Inflation Differential" (frame 17).*

The real exchange rate:

$$\Delta e^r_t = \Delta e_t + \pi^f_t - \pi_t$$

For Denmark under the peg, $\Delta e_t \approx 0$, so:

$$\Delta e^r_t \approx \pi^f_t - \pi_t$$

When Danish inflation is **below** euro-area inflation, the real exchange rate rises and Danish goods become relatively cheaper (competitiveness improves). The notebook downloads year-on-year HICP from FRED and cumulates monthly differentials into a competitiveness index.


In [ ]:
# === Download HICP data (falls back to sample data when offline) ===
fallback   = fallback_hicp_data()
series_ids = {'Denmark': 'CP0000DKM086NEST', 'Euro area': 'CP0000EZ19M086NEST'}

hicp, statuses = {}, []
for label, sid in series_ids.items():
    s, status = fetch_fred_series(sid, fallback)
    hicp[label] = s
    statuses.append(status)

hicp_df = pd.concat(hicp, axis=1).dropna()
pi_df   = hicp_df.apply(yoy_inflation).dropna()
pi_df   = pi_df[pi_df.index >= '2021-01-01']
statuses


In [ ]:
# === Year-on-year HICP inflation ===
fig = plt.figure(figsize=(8.5, 4.2))
ax  = fig.add_subplot(1, 1, 1)

ax.plot(pi_df.index, pi_df['Denmark'],   lw=2, ls='-',  color=colors[0], label='Denmark')
ax.plot(pi_df.index, pi_df['Euro area'], lw=2, ls='--', color=colors[1], label='Euro area')
ax.axhline(2.0, lw=1, ls=':', color='grey', alpha=0.7, label='2% reference')
ax.axhline(0.0, lw=0.8, color='grey', alpha=0.4)

ax.text(pi_df.index[-1], pi_df['Denmark'].iloc[-1] + 0.25,
        f"{pi_df['Denmark'].iloc[-1]:.1f}%", fontsize=9, color=colors[0])
ax.text(pi_df.index[-1], pi_df['Euro area'].iloc[-1] + 0.25,
        f"{pi_df['Euro area'].iloc[-1]:.1f}%", fontsize=9, color=colors[1])

ax.set_title('Year-on-year HICP inflation: Denmark vs euro area (2021 onward)')
ax.set_xlabel('Date')
ax.set_ylabel('Year-on-year change (percent)')
ax.legend(loc='upper left')
fig.tight_layout()
print('Source: Eurostat via FRED (CP0000DKM086NEST, CP0000EZ19M086NEST).')


**What is done:** Year-on-year HICP inflation is downloaded and plotted from 2021 onward. The most recent values are labelled on the right.

**Why it is useful:** This is the **real-world input** to the formula $\Delta e^r \approx \pi^f - \pi$. Whenever the Danish line lies below the euro-area line, Denmark is gaining price competitiveness. The 2022 energy shock hit both economies, but the *gap* between them determines the real exchange rate, not their common level.

**How to interpret the result:** Periods where Denmark is below the euro area are periods of improving Danish competitiveness (real exchange rate rising). By late 2023 and into 2024, Danish HICP settled clearly below the euro area, implying a gradual real depreciation that mechanically supported net exports without any nominal exchange-rate movement.


In [ ]:
# === Approximate DKK/EUR real exchange rate index ===
monthly_diff = (pi_df['Euro area'] - pi_df['Denmark']) / 12.0
rer_index    = 100.0 + monthly_diff.cumsum()

fig = plt.figure(figsize=(8.5, 4.2))
ax  = fig.add_subplot(1, 1, 1)

ax.plot(rer_index.index, rer_index, lw=2, ls='-', color=colors[0],
        label='Approximate real exchange rate index')
ax.axhline(100.0, lw=1, ls='--', color='grey', label='Base (Jan 2021 = 100)')
ax.axvspan(pd.Timestamp('2022-01-01'), pd.Timestamp('2023-06-01'),
           alpha=0.10, color='red', label='High euro-area inflation episode (2022)')

ax.set_title('Approximate DKK/EUR real exchange rate index (nominal rate held fixed)')
ax.set_xlabel('Date')
ax.set_ylabel('Index (Jan 2021 = 100)')
ax.legend(loc='lower right')
fig.tight_layout()
print('Source: own calculation from Eurostat HICP via FRED. DKK 7.46038 per EUR assumed throughout.')


**What is done:** Monthly differentials $(\pi^f - \pi)/12$ are cumulated to form a competitiveness index from January 2021.

**Why it is useful:** This translates the abstract formula $\Delta e^r = \pi^f - \pi$ into a concrete measurable trajectory. The shaded area marks the 2022 energy shock, when euro-area inflation exceeded Danish inflation by several percentage points per year, temporarily boosting Danish competitiveness even though the nominal exchange rate never moved.

**How to interpret the result:** An index above 100 means Danish goods are cheaper relative to euro-area goods than in January 2021. Any dip back toward 100 reflects episodes where Denmark ran relatively higher inflation. **Key exam point:** under a peg, this slow real-exchange-rate drift is the only demand-side equilibrating mechanism, replacing nominal depreciation.


### 4.1. <a id='toc4_1_'></a>[Try it yourself](#toc0_)

**Task A.** Change the sample start date from `'2021-01-01'` to `'2022-06-01'` (the inflation peak). Does the cumulative real depreciation become larger or smaller? Which sub-period drives most of the competitiveness gain?

**Task B.** Compute the **average annual inflation differential** $\pi^f - \pi$ over the full sample. Is Denmark systematically below or above the euro area? What does relative PPP predict about the real exchange rate trend over the long run?

**Task C.** Suppose Denmark ran 2 percentage points higher inflation than the euro area in every month of 2022. Modify `monthly_diff` by subtracting `2/1200` in those months (`pi_df.index.year == 2022`), then replot. By how many index points would competitiveness have eroded by end-2023?


In [ ]:
# write your code here


## 5. <a id='toc5_'></a>[Marshall-Lerner and the J-curve](#toc0_)

*Corresponds to slides: "Marshall-Lerner Condition" (frame 21) and "The J-Curve" (frame 22).*

A real depreciation raises net exports **if and only if**:

$$\varepsilon_X + \varepsilon_M > 1 \qquad \text{(Marshall-Lerner condition)}$$

In the short run, quantities are fixed by contracts and habits. The price effect is immediate: imports cost more in domestic currency. Export volumes adjust only over months as foreign buyers switch suppliers. This generates the **J-curve**: the trade balance first worsens, then improves.


In [ ]:
# === J-curve: trade-balance response to a 10% real depreciation ===
T_j = 30
dep = 0.10

el_fast, tb_fast = simulate_j_curve(T_j, dep,
    eps_x_start=0.55, eps_m_start=0.60,
    eps_x_long=0.75,  eps_m_long=0.80, speed=0.40)

el_slow, tb_slow = simulate_j_curve(T_j, dep,
    eps_x_start=0.15, eps_m_start=0.20,
    eps_x_long=0.75,  eps_m_long=0.80, speed=0.18)

period_j = np.arange(T_j)

fig = plt.figure(figsize=(12, 4.2))
ax1 = fig.add_subplot(1, 2, 1)
ax2 = fig.add_subplot(1, 2, 2)

# panel a: trade balance
ax1.plot(period_j, 100 * tb_fast, lw=2, ls='-',  color=colors[0], label='M-L holds immediately')
ax1.plot(period_j, 100 * tb_slow, lw=2, ls='--', color=colors[1], label='J-curve case')
ax1.fill_between(period_j, 100 * tb_slow, 0,
                 where=(tb_slow < 0), alpha=0.12, color=colors[1], label='Initial deterioration')
ax1.axhline(0.0, lw=1, ls=':', color='grey')
ax1.set_title('(a) Trade-balance change after a 10% real depreciation')
ax1.set_xlabel('Period after depreciation')
ax1.set_ylabel('Change in trade balance (% of initial trade)')
ax1.legend(loc='lower right')

# panel b: elasticity sums
ax2.plot(period_j, el_fast, lw=2, ls='-',  color=colors[0], label='Fast quantity response')
ax2.plot(period_j, el_slow, lw=2, ls='--', color=colors[1], label='Slow quantity response')
ax2.axhline(1.0, lw=1.5, ls=':', color='black', label='M-L threshold = 1')
ax2.set_title('(b) Export + import elasticity sum over time')
ax2.set_xlabel('Period after depreciation')
ax2.set_ylabel(r'$\varepsilon_X + \varepsilon_M$')
ax2.legend(loc='lower right')

fig.tight_layout()
print(f'Depreciation = {100*dep:.0f}%. Elasticities converge exponentially from initial to long-run values.')


**What is done:** Two scenarios simulate a 10% real depreciation. In the "fast" case, initial elasticities already exceed 1 (M-L holds immediately). In the "slow" case, initial elasticities sum to 0.35, below threshold, producing a J-curve.

**Why it is useful:** The **J-curve** matters for policy evaluation. The shaded area is the cumulative trade-balance cost before the volume response kicks in. For Denmark, pharmaceutical exports involve multi-year contracts that respond slowly to relative prices, contributing to J-curve dynamics. The **Marshall-Lerner condition** is a medium-run statement, not a period-by-period law.

**How to interpret the result:** Panel (b) shows when the elasticity sum crosses 1, which is exactly when the trade balance turns positive in panel (a). With slow adjustment, the crossing takes much longer and the trough is deeper. The shaded area in panel (a) captures the total temporary resource transfer abroad before the volume response dominates.


### 5.1. <a id='toc5_1_'></a>[Try it yourself](#toc0_)

**Task A.** Increase `speed` from `0.18` to `0.55` in the J-curve case. By how many periods does the trade balance return to zero earlier? What real-world factor determines `speed` (think: contract lengths, consumer habits, production lead times)?

**Task B.** Change `dep` from `0.10` to `0.05` (a 5% depreciation). Does the J-curve still appear in the slow case? Is the shaded area proportionally smaller, the same, or larger as a share of the depreciation size?

**Task C.** Simulate a **permanent M-L failure**: set `eps_x_long = 0.40` and `eps_m_long = 0.45` (long-run sum = 0.85 < 1). What does the trade balance look like after 30 periods? What type of economy (think: heavy import dependence with few close domestic substitutes) might face this situation?


In [ ]:
# write your code here


## 6. <a id='toc6_'></a>[Open-economy AD adjustment under a peg](#toc0_)

*Corresponds to slides: "Open-Economy IS Curve" (frame 27) and "Adjustment From a Recession" (frame 32).*

Under a fixed nominal exchange rate, the model in deviations from long-run equilibrium is:

$$\hat{y}_t = \frac{\beta_1 \, e^r_{t-1} + z_t}{1 + \beta_1 \gamma}, \qquad \hat{\pi}_t = \gamma \, \hat{y}_t, \qquad e^r_t = e^r_{t-1} - \hat{\pi}_t$$

A negative shock $z_0 < 0$ opens a recession. Lower output reduces inflation, which raises the real exchange rate, which gradually boosts net exports. Recovery is **export-led** and requires no policy action.


In [ ]:
# === Baseline AD-AS adjustment after a negative demand shock ===
ad = simulate_fixed_rate_adjustment(
    T=30, beta1=par['beta1'], gamma=par['gamma'], z0=par['z_shock'])

fig = plt.figure(figsize=(8.5, 4.5))
ax  = fig.add_subplot(1, 1, 1)

ax.plot(ad['period'], ad['yhat'],  lw=2, ls='-',  color=colors[0],
        label=r'Output gap $\hat{y}_t$')
ax.plot(ad['period'], ad['pihat'], lw=2, ls='--', color=colors[1],
        label=r'Inflation gap $\hat{\pi}_t$')
ax.plot(ad['period'], ad['er'],    lw=2, ls=':',  color=colors[2],
        label=r'Real exchange rate $e^r_t$')
ax.axhline(0.0, lw=1, color='grey')

ax.annotate(f'Shock $z_0 = {par["z_shock"]:.1f}$',
            xy=(0, ad['yhat'].iloc[0]),
            xytext=(3, ad['yhat'].iloc[0] - 0.12),
            arrowprops=dict(arrowstyle='->', color='grey'), fontsize=9, color='grey')

ax.set_title(f'Recession and export-led recovery under a peg'
             f' (beta1={par["beta1"]}, gamma={par["gamma"]})')
ax.set_xlabel('Period')
ax.set_ylabel('Deviation from long-run equilibrium')
ax.legend(loc='lower right')
fig.tight_layout()


**What is done:** A one-period negative demand shock of size $z_0 = -1$ hits in period 0. The model is solved forward with a fixed nominal rate and no additional policy action.

**Why it is useful:** This is the core adjustment mechanism for Denmark under the peg, and the answer to **Socrative Q3**. The economy cannot use nominal depreciation or an independent interest rate. The adjustment chain is: recession $\to$ $\pi < \pi^f$ $\to$ real exchange rate rises $\to$ net exports increase $\to$ AD shifts right $\to$ output returns to potential.

**How to interpret the result:** The output gap (solid) is deepest on impact and converges monotonically to zero. The inflation gap (dashed) is proportional to the output gap with slope $\gamma$. The real exchange rate (dotted) accumulates as long as inflation is below normal, then gradually normalises. Recovery speed is governed by $\beta_1 \gamma$: a strong trade channel and flexible prices together imply fast adjustment.


In [ ]:
# === Sensitivity to beta1: how strongly does the trade channel support recovery? ===
beta_vals = [0.20, 0.60, 0.80, 1.50]

fig = plt.figure(figsize=(12, 4.2))
ax1 = fig.add_subplot(1, 2, 1)
ax2 = fig.add_subplot(1, 2, 2)

for i, b1 in enumerate(beta_vals):
    path = simulate_fixed_rate_adjustment(T=30, beta1=b1, gamma=par['gamma'], z0=par['z_shock'])
    ax1.plot(path['period'], path['yhat'], lw=2, color=colors[i],
             label=f'$\\beta_1 = {b1:.2f}$')
    ax2.plot(path['period'], path['er'],   lw=2, color=colors[i],
             label=f'$\\beta_1 = {b1:.2f}$')

for ax, title, yl in [
        (ax1, '(a) Output gap: higher trade channel, faster recovery',
               r'Output gap $\hat{y}_t$'),
        (ax2, '(b) Real exchange rate accumulation',
               r'Real exchange rate $e^r_t$')]:
    ax.axhline(0.0, lw=1, ls=':', color='grey')
    ax.set_title(title)
    ax.set_xlabel('Period')
    ax.set_ylabel(yl)
    ax.legend(loc='lower right')

fig.tight_layout()


**What is done:** The same negative shock is simulated for four values of $\beta_1$.

**Why it is useful:** $\beta_1$ is the **key structural parameter for adjustment under a peg**. It captures trade openness and price elasticities combined. A high $\beta_1$ means the Marshall-Lerner condition holds strongly: real depreciation quickly raises demand. A low $\beta_1$ means the economy depends on slow price-level adjustment. This explains why structurally similar countries can have very different recovery times under the same exchange-rate regime.

**How to interpret the result:** Panel (a) shows that with $\beta_1 = 1.50$ the output gap closes within five to six periods; with $\beta_1 = 0.20$ it takes more than twenty. Panel (b) shows the real exchange rate must accumulate a much larger stock of competitiveness when $\beta_1$ is low, because each unit of $e^r$ translates into little additional demand.


In [ ]:
# === Positive export boom versus negative recession: symmetric adjustment ===
fig = plt.figure(figsize=(12, 4.2))
ax1 = fig.add_subplot(1, 2, 1)
ax2 = fig.add_subplot(1, 2, 2)

for z0, label, col, ls in [
        (par['z_shock'],   'Negative shock (recession)',   colors[0], '-'),
        (-par['z_shock'],  'Positive shock (export boom)', colors[1], '--'),
]:
    path = simulate_fixed_rate_adjustment(T=30, beta1=par['beta1'], gamma=par['gamma'], z0=z0)
    ax1.plot(path['period'], path['yhat'], lw=2, ls=ls, color=col, label=label)
    ax2.plot(path['period'], path['er'],   lw=2, ls=ls, color=col, label=label)

for ax, title, yl in [
        (ax1, '(a) Output gap: symmetric in and out',
               r'Output gap $\hat{y}_t$'),
        (ax2, '(b) Real exchange rate: falls (appreciates) in a boom',
               r'Real exchange rate $e^r_t$')]:
    ax.axhline(0.0, lw=1, ls=':', color='grey')
    ax.set_title(title)
    ax.set_xlabel('Period')
    ax.set_ylabel(yl)
    ax.legend(loc='upper right')

fig.tight_layout()
print('Positive shock z0 = +1.0, symmetric to the recession. Mimics a concentrated export-sector boom.')


**What is done:** The model is run for a positive demand shock of the same size, symmetric to the recession.

**Why it is useful:** This is the **Novo Nordisk scenario**. Pharmaceutical sales rose from roughly 1% of Danish GDP in the early 1990s to 8.3% in 2023, and pharmaceuticals reached about 24% of Danish goods exports in 2024. A sustained positive shock to export demand pushes output above potential, raises inflation above the euro area, erodes the real exchange rate (real appreciation), and gradually dampens net exports. **The peg constrains adjustment in a boom as much as in a recession**: the Nationalbank cannot lean against the boom by raising rates independently.

**How to interpret the result:** Panel (a) confirms the exact symmetry. Panel (b) shows the real exchange rate *falling* (real appreciation) during the boom, the opposite of the recession. In practice, Denmark's pharmaceutical boom created precisely this inflationary pressure and competitiveness erosion, with the real exchange rate doing the equilibrating work that monetary policy cannot do.


### 6.1. <a id='toc6_1_'></a>[Try it yourself](#toc0_)

**Task A.** Change `par['gamma']` from `0.50` to `0.15` (flatter AS curve, stickier wages and prices). How does this affect: (i) the impact depth of the recession; (ii) the number of periods to 90% recovery; (iii) the peak real exchange rate accumulation? What does this tell you about the importance of wage flexibility for adjustment under a peg?

**Task B.** Simulate a **persistent** demand shock by writing a short loop where $z_t = -1 \times 0.80^t$ for the first 10 periods, zero after. Use `simulate_fixed_rate_adjustment` once per period or write a new version that accepts a `z` array. Does the output gap converge faster or slower than in the one-shot case?

**Task C.** Using the `beta_vals` plot, compute the number of periods until $\hat{y}$ first exceeds $-0.10$ (90% recovery) for $\beta_1 = 0.20$ and $\beta_1 = 1.50$ using `np.argmax(path['yhat'] > -0.10)`. What is the difference in periods? What fiscal or structural policy could a pegged country use to speed recovery when $\beta_1$ is low?


In [ ]:
# write your code here


## 7. <a id='toc7_'></a>[Group exercise](#toc0_)

Work in pairs or small groups. Write brief answers in a shared document.

**Question 1 (UIP + trilemma).** The ECB held its deposit rate at 4.0% from September 2023 to June 2024. Use `uip_rate` to verify: what Danish overnight rate does UIP predict when the peg is fully credible and $\Delta e^e = 0$? Now suppose a rumour causes markets to price in a 0.5% expected depreciation. What rate would the Nationalbank need to set? Is that within or outside the ERM II band logic?

**Question 2 (Real exchange rate).** From the HICP plot in Section 4, identify one 3-month window where Denmark gained competitiveness and one where it lost. Translate both into approximate $\Delta e^r$ values using the formula. Which one matters more for Danish net exports?

**Question 3 (J-curve).** A classmate argues: "The J-curve proves that depreciation is bad policy because the trade balance gets worse first." Write three sentences explaining why this argument is incomplete, using the elasticity panel (b) from Section 5.

**Question 4 (AD mechanism).** Connect the lecture 9 opening hook ("the quietest krone market since 1982") to the AD adjustment plot in Section 6. Why does a large current-account surplus (8.5% of GDP in 2024) reduce pressure on the Nationalbank to intervene during calm periods? Which parameter in the model captures the strength of this buffer?


## 8. <a id='toc8_'></a>[Summary](#toc0_)

| Equation | Meaning |
|---|---|
| $i_t = i_t^f + \Delta e^e_{t+1}$ | UIP: domestic rate = foreign rate + expected depreciation |
| $\Delta e^r_t = \Delta e_t + \pi^f_t - \pi_t$ | Real depreciation via nominal change or inflation differential |
| $\Delta e_t = \pi_t - \pi^f_t$ (long run) | Relative PPP: nominal rate must drift to keep competitiveness stable |
| $\varepsilon_X + \varepsilon_M > 1$ | Marshall-Lerner: depreciation raises net exports in the medium run |
| $y_t - \bar y = \beta_1 e^r_t - \beta_2(r_t - \bar r) + z_t$ | Open-economy IS: demand depends on competitiveness and real rate |
| $e^r_t = e^r_{t-1} - \hat{\pi}_t$ | Under a peg, real rate rises when domestic inflation is below foreign |

**Key mechanism under a peg:** recession $\to$ $\pi < \pi^f$ $\to$ $e^r \uparrow$ $\to$ net exports rise $\to$ AD right $\to$ recovery. Runs symmetrically for a boom.

**Danish facts to remember:**
- ERM II central rate: DKK 7.46038 per EUR, band $\pm 2.25\%$, peg in place since 1982
- Late 2023 to February 2025: longest FX-intervention-free period since 1982 (Danmarks Nationalbank Annual Report 2024)
- Current-account surplus approximately 8.5% of GDP in 2024; pharmaceuticals approximately 24% of Danish goods exports

**Socrative room:** MACROECONOMICSB
